# Predict Progress

## Import Libraries

In [6]:
import pandas as pd
import pickle
import numpy as np
import onnxruntime as ort

## Load Models

In [7]:
# ==============================================================================
# GLOBAL METADATA & ONNX INITIALIZATION
# ==============================================================================

# 1. Load Metadata (Encoders & Feature List)
# We still load the pickle file JUST for the LabelEncoders and Feature Map.
with open('../../models/model_progress.pickle', 'rb') as f:
    data_metadata = pickle.load(f)

encoders = data_metadata['encoders']
feature_order = data_metadata['features']

# 2. Load ONNX Models
# Define the 9 numeric targets trained in the modeling notebook
targets_num = [
    'Weight_kg', 'Body_Fat_Percentage_y', 'Daily_Calories', 
    'Daily_Water_ml', 'Target_Protein_g', 'Target_Carbs_g', 'Target_Fat_g',
    'Limit_Sugar_g', 'Target_Fiber_g'
]

# Initialize ONNX Inference Sessions for each target
ort_sessions = {}
for target in targets_num:
    model_path = f"../../models/onnx/{target}.onnx"
    # Create an inference session for each model
    ort_sessions[target] = ort.InferenceSession(model_path)

print(f"✅ Loaded {len(ort_sessions)} ONNX models and metadata successfully.")

c:\Users\rachm\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Loaded 9 ONNX models and metadata successfully.


## Predict Function

In [8]:
# ==============================================================================
# PREDICTION ENGINE (WEEKLY PROGRESS - ONNX VERSION)
# ==============================================================================

def predict_complete_progress(user_data, week_num):
    """
    Predicts a user's physical status and nutritional requirements for a specific week 
    using serialized ONNX machine learning models.

    This engine processes the user's baseline data, encodes categorical features, 
    and performs parallel inference across multiple regression models. It also 
    applies post-processing heuristics to derive categorical targets not covered 
    by the ML models.

    ---------------------------------------------------------------------------
    Args:
        user_data (dict): A dictionary containing the user's initial baseline profile.
                          Required keys include demographics ('Age', 'Gender', 'Height_cm', 
                          'Initial_Weight_kg'), body composition ('Body_Fat_Percentage_x'), 
                          lifestyle ('Workout_Frequency', 'Average_Duration_Minutes'), 
                          and sports flags ('Badminton', 'Football', etc.).
        week_num (int): The target week number in the simulation timeline (e.g., 1 to 12).
                        Acts as the primary time-series variable for the predictions.

    Returns:
        dict: A comprehensive dictionary of predicted metrics for the requested week, 
              including 'Weight_kg', 'BMI', 'Daily_Calories', and macronutrient targets.
    """

    # --- A. FEATURE ENGINEERING (Calculate Derived Baseline Metrics) ---
    height_m = user_data['Height_cm'] / 100
    initial_bmi = round(user_data['Initial_Weight_kg'] / (height_m ** 2), 2)
    
    # Determine the Initial BMI Category based on WHO standards
    if initial_bmi < 18.5: 
        initial_bmi_cat = 'Underweight'
    elif initial_bmi < 25.0: 
        initial_bmi_cat = 'Normal'
    elif initial_bmi < 30.0: 
        initial_bmi_cat = 'Overweight'
    else: 
        initial_bmi_cat = 'Obese'
    
    # --- B. DATA ENCODING (Text -> Numeric) ---
    # Transform categorical strings into numeric representations using pre-fitted encoders
    gender_code = encoders['Gender'].transform([user_data['Gender']])[0]
    goal_code = encoders['Goal'].transform([user_data['Goal']])[0]
    level_code = encoders['level'].transform([user_data['level']])[0]
    bmi_cat_code = encoders['BMI_Category_x'].transform([initial_bmi_cat])[0]

    # --- C. CONSTRUCT INPUT ARRAY (ONNX FORMAT) ---
    # CRITICAL: The feature order must strictly match the training phase blueprint
    input_row = [
        user_data['Age'], 
        gender_code, 
        user_data['Height_cm'], 
        user_data['Initial_Weight_kg'],
        bmi_cat_code, 
        user_data.get('Body_Fat_Category', 0),
        goal_code, 
        user_data['Workout_Frequency'], 
        user_data['Average_Duration_Minutes'], 
        level_code,
        user_data.get('Badminton', 0), 
        user_data.get('Football', 0), 
        user_data.get('Basketball', 0),
        user_data.get('Volleyball', 0), 
        user_data.get('Swim', 0),
        week_num # Dynamic time variable
    ]
    
    # ONNX runtime strictly requires a 2D NumPy array with a float32 data type
    input_array = np.array([input_row], dtype=np.float32)
    
    # --- D. ONNX PREDICTION LOOP ---
    results = {}
    
    # Iterate through all loaded ONNX sessions (Weight, Calories, Macros, etc.)
    for target, sess in ort_sessions.items():
        # Dynamically fetch the expected input node name for the current model
        input_name = sess.get_inputs()[0].name
        
        # Execute ONNX inference
        pred = sess.run(None, {input_name: input_array})[0]
        
        # Extract the scalar prediction and map it to the target metric
        results[target] = float(pred[0])
        
    # --- E. POST-PROCESSING (Heuristic Calculations) ---
    # Calculate categorical metrics that were excluded from the numeric ML training phase
    
    # 1. Calculate Predicted BMI and its corresponding Category
    pred_bmi = results['Weight_kg'] / (height_m ** 2)
    results['BMI'] = pred_bmi
    
    if pred_bmi < 18.5: 
        results['BMI_Category_y'] = 'Underweight'
    elif pred_bmi < 25.0: 
        results['BMI_Category_y'] = 'Normal'
    elif pred_bmi < 30.0: 
        results['BMI_Category_y'] = 'Overweight'
    else: 
        results['BMI_Category_y'] = 'Obese'
        
    # 2. Determine Meal Frequency based on predicted daily caloric expenditure
    pred_cals = results['Daily_Calories']
    if pred_cals < 1500: 
        results['Meal_Frequency'] = 3
    elif pred_cals < 2200: 
        results['Meal_Frequency'] = 4
    else: 
        results['Meal_Frequency'] = 5
            
    return results

## Simulation & Testing

In [9]:
# ==============================================================================
# SIMULATION & TESTING (EXECUTION)
# ==============================================================================

input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Initial_Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Volleyball': 0, 
    'Swim': 0
}

print(f"User: Pria, 25th, 60kg -> Goal: Muscle Gain")

for minggu in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    
    output = predict_complete_progress(input_user, week_num=minggu)
    
    print(f"\n{'='*10} MINGGU KE-{minggu} {'='*10}")
    
    print(f"[Physique]")
    print(f"  Berat Badan     : {output['Weight_kg']:.2f} kg")
    print(f"  BMI             : {output['BMI']:.2f} ({output['BMI_Category_y']})")
    print(f"  Body Fat        : {output['Body_Fat_Percentage_y']:.1f}%")
    
    print(f"[Daily Nutrition]")
    print(f"  Kalori          : {output['Daily_Calories']:.0f} kkal")
    print(f"  Air Minum       : {output['Daily_Water_ml']:.0f} ml")
    print(f"  Gula (Limit)    : {output['Limit_Sugar_g']:.1f} g")
    print(f"  Meal Freq       : {output['Meal_Frequency']}x / hari")
    
    print(f"[Macro Nutrition]")
    print(f"  Protein         : {output['Target_Protein_g']:.1f} g")
    print(f"  Karbo           : {output['Target_Carbs_g']:.1f} g")
    print(f"  Lemak           : {output['Target_Fat_g']:.1f} g")
    print(f"  Serat           : {output['Target_Fiber_g']:.1f} g")

User: Pria, 25th, 60kg -> Goal: Muscle Gain

========== MINGGU KE-1 ==========
[Physique]
  Berat Badan     : 60.87 kg
  BMI             : 19.88 (Normal)
  Body Fat        : 13.5%
[Daily Nutrition]
  Kalori          : 2688 kkal
  Air Minum       : 2702 ml
  Gula (Limit)    : 66.7 g
  Meal Freq       : 5x / hari
[Macro Nutrition]
  Protein         : 205.0 g
  Karbo           : 333.1 g
  Lemak           : 60.3 g
  Serat           : 36.9 g

========== MINGGU KE-2 ==========
[Physique]
  Berat Badan     : 60.98 kg
  BMI             : 19.91 (Normal)
  Body Fat        : 13.4%
[Daily Nutrition]
  Kalori          : 2688 kkal
  Air Minum       : 2702 ml
  Gula (Limit)    : 66.7 g
  Meal Freq       : 5x / hari
[Macro Nutrition]
  Protein         : 205.0 g
  Karbo           : 333.1 g
  Lemak           : 60.3 g
  Serat           : 36.9 g

========== MINGGU KE-3 ==========
[Physique]
  Berat Badan     : 61.08 kg
  BMI             : 19.94 (Normal)
  Body Fat        : 13.1%
[Daily Nutrition]
  Kalori

C:\Users\rachm\AppData\Local\Temp\ipykernel_18720\3667343124.py:87: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  results[target] = float(pred[0])
C:\Users\rachm\AppData\Local\Temp\ipykernel_18720\3667343124.py:87: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  results[target] = float(pred[0])
C:\Users\rachm\AppData\Local\Temp\ipykernel_18720\3667343124.py:87: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  results[target] = float(pred[0])
C:\Users\rachm\AppData\Local